In [9]:
import os
import json
import urllib.request
import urllib.parse
import streamlit as st
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_core.messages import ToolMessage, HumanMessage, AIMessage
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
import wikipedia

In [2]:
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:


@tool
def get_live_weather(city: str) -> str:
    """Fetch real-time weather data for a specific city name."""
    encoded_city = urllib.parse.quote(city)
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={encoded_city}&count=1&language=en&format=json"
    try:
        with urllib.request.urlopen(geo_url) as geo_response:
            geo_data = json.loads(geo_response.read().decode())
            if not geo_data.get("results"):
                return f"Could not find coordinates for city: {city}"
            loc = geo_data["results"][0]
            lat, lon, resolved = loc["latitude"], loc["longitude"], loc["name"]

        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        with urllib.request.urlopen(weather_url) as weather_response:
            weather_data = json.loads(weather_response.read().decode())
            current = weather_data["current_weather"]
            return f"Temperature in {resolved} is {current['temperature']}°C."
    except Exception as e:
        return f"Weather API Error: {str(e)}"
# 1. Fix Wikipedia by overriding the blocked User-Agent
wikipedia.set_user_agent("research-assistant/1.0 ")

wikipedia_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=1000)
)
wikipedia_tool.description = "A wrapper around Wikipedia. Use strictly for finding factual information about people, places, companies, or historical events. Provide only the exact subject noun as the query."

# 2. Fix ArXiv by forcing strict keyword usage in the docstring
@tool
def search_arxiv(query: str) -> str:
    """Search the ArXiv academic database for scientific papers.
    CRITICAL: The query must be STRICTLY keywords (e.g., 'quantum mechanics', 'transformers'). 
    DO NOT use conversational language. 
    """
    try:
        arxiv = ArxivAPIWrapper(top_k_results=3, doc_content_chars_max=1500)
        result = arxiv.run(query)
        if not result or "No good Arxiv Result" in result:
            return f"Search failed. The query '{query}' returned no results. Try broader keywords."
        return result
    except Exception as e:
        return f"ArXiv API Error: {str(e)}"

tools = [search_arxiv, get_live_weather,wikipedia_tool]

In [4]:
t_llm=llm.bind_tools(tools)

In [6]:
rsp=t_llm.invoke("what is attention all you need paper and what it about use the tool?")

In [8]:
rsp.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  search_arxiv (fc_e0adf9b9-12b7-4cca-a9a9-7bbde091f2d1)
 Call ID: fc_e0adf9b9-12b7-4cca-a9a9-7bbde091f2d1
  Args:
    query: Attention Is All You Need
